---
# ***1. SELECT + WHERE — filter rows***
---

In [29]:
import sqlite3
import pandas as pd
df = pd.read_csv('imdb_top_1000_cleaned.csv')
conn = sqlite3.connect('imdb.db')
df.to_sql('movies', conn, if_exists='replace', index=False)
result = pd.read_sql_query("SELECT * FROM movies LIMIT 5", conn)
result

,Poster_Link,Series_Title,Released_Year,Certificate,Runtime,Genre,IMDB_Rating,Overview,Meta_score,Director,Star1,Star2,Star3,Star4,No_of_Votes,Gross,Decade
0,https://m.media-amazon.com/images/M/MV5BMDFkYT...,The Shawshank Redemption,1994.0,A,142,Drama,9.3,Two imprisoned men bond over a number of years...,80.0,Frank Darabont,Tim Robbins,Morgan Freeman,Bob Gunton,William Sadler,2343110,28341469.0,1990.0
1,https://m.media-amazon.com/images/M/MV5BM2MyNj...,The Godfather,1972.0,A,175,"Crime, Drama",9.2,An organized crime dynasty's aging patriarch t...,100.0,Francis Ford Coppola,Marlon Brando,Al Pacino,James Caan,Diane Keaton,1620367,134966411.0,1970.0
2,https://m.media-amazon.com/images/M/MV5BMTMxNT...,The Dark Knight,2008.0,UA,152,"Action, Crime, Drama",9.0,When the menace known as the Joker wreaks havo...,84.0,Christopher Nolan,Christian Bale,Heath Ledger,Aaron Eckhart,Michael Caine,2303232,534858444.0,2000.0
3,https://m.media-amazon.com/images/M/MV5BMWMwMG...,The Godfather: Part II,1974.0,A,202,"Crime, Drama",9.0,The early life and career of Vito Corleone in ...,90.0,Francis Ford Coppola,Al Pacino,Robert De Niro,Robert Duvall,Diane Keaton,1129952,57300000.0,1970.0
4,https://m.media-amazon.com/images/M/MV5BMWU4N2...,12 Angry Men,1957.0,U,96,"Crime, Drama",9.0,A jury holdout attempts to prevent a miscarria...,96.0,Sidney Lumet,Henry Fonda,Lee J. Cobb,Martin Balsam,John Fiedler,689845,4360000.0,1950.0


In [8]:
query = """
SELECT Series_Title, IMDB_Rating, Released_Year	
FROM movies
WHERE IMDB_Rating >= 8.5
ORDER BY IMDB_Rating DESC
"""
pd.read_sql_query(query, conn)

,Series_Title,IMDB_Rating,Released_Year
0,The Shawshank Redemption,9.3,1994.0
1,The Godfather,9.2,1972.0
2,The Dark Knight,9.0,2008.0
3,The Godfather: Part II,9.0,1974.0
4,12 Angry Men,9.0,1957.0
5,The Lord of the Rings: The Return of the King,8.9,2003.0
6,Pulp Fiction,8.9,1994.0
7,Schindler's List,8.9,1993.0
8,Inception,8.8,2010.0
9,Fight Club,8.8,1999.0


---
# ***2. GROUP BY + Aggregates — summarize by category***
---

In [10]:
query = """
SELECT Genre, COUNT(*) as movie_count, ROUND(AVG(IMDB_Rating), 2) as avg_rating
FROM movies
GROUP BY Genre
ORDER BY IMDB_Rating DESC
LIMIT 10
"""
pd.read_sql_query(query, conn)

,Genre,movie_count,avg_rating
0,Drama,85,7.98
1,"Crime, Drama",26,8.16
2,"Action, Crime, Drama",30,7.88
3,"Biography, Drama, History",28,8.02
4,"Action, Adventure, Drama",14,8.15
5,Western,4,8.35
6,"Drama, Romance",37,7.94
7,"Action, Adventure, Sci-Fi",21,7.93
8,"Biography, Crime, Drama",16,7.92
9,"Action, Sci-Fi",3,8.40


---
# ***3. LIKE — pattern matching on strings***
---

In [14]:
query = """
SELECT Series_Title, Genre, IMDB_Rating
FROM movies
WHERE Genre LIKE '%DRAMA%'
ORDER BY IMDB_Rating DESC
LIMIT 10
"""
pd.read_sql_query(query, conn)

,Series_Title,Genre,IMDB_Rating
0,The Shawshank Redemption,Drama,9.3
1,The Godfather,"Crime, Drama",9.2
2,The Dark Knight,"Action, Crime, Drama",9.0
3,The Godfather: Part II,"Crime, Drama",9.0
4,12 Angry Men,"Crime, Drama",9.0
5,The Lord of the Rings: The Return of the King,"Action, Adventure, Drama",8.9
6,Pulp Fiction,"Crime, Drama",8.9
7,Schindler's List,"Biography, Drama, History",8.9
8,Fight Club,Drama,8.8
9,The Lord of the Rings: The Fellowship of the Ring,"Action, Adventure, Drama",8.8


---
# ***4. INNER JOIN — combine two tables***
---

In [34]:
directors_df = pd.DataFrame({
    'Director' : ['Christopher Nolan', 'Frank Darabont', 'Francis Ford Coppola', 'David Fincher'],
    'Nationality' : ['British', 'American', 'American', 'American']
})
directors_df.to_sql('directors', conn, if_exists='replace', index=False)
result = pd.read_sql_query("SELECT * FROM directors LIMIT 5", conn)
result

,Director,Nationality
0,Christopher Nolan,British
1,Frank Darabont,American
2,Francis Ford Coppola,American
3,David Fincher,American


In [35]:
query = """
SELECT m.Series_Title, m.IMDB_Rating, d.Nationality
FROM movies m
INNER JOIN directors d ON m.Director = d.Director
ORDER BY m.IMDB_Rating DESC
"""
pd.read_sql_query(query, conn)

,Series_Title,IMDB_Rating,Nationality
0,The Shawshank Redemption,9.3,American
1,The Godfather,9.2,American
2,The Dark Knight,9.0,British
3,The Godfather: Part II,9.0,American
4,Inception,8.8,British
5,Fight Club,8.8,American
6,Interstellar,8.6,British
7,The Green Mile,8.6,American
8,Se7en,8.6,American
9,The Prestige,8.5,British


---
# ***5. Subquery — query inside a query***
---

In [43]:
query = """
SELECT Series_Title, IMDB_Rating, Director
FROM movies
WHERE IMDB_Rating > (SELECT AVG(IMDB_Rating) FROM movies)
ORDER BY IMDB_Rating DESC
LIMIT 10
"""
pd.read_sql_query(query, conn)

,Series_Title,IMDB_Rating,Director
0,The Shawshank Redemption,9.3,Frank Darabont
1,The Godfather,9.2,Francis Ford Coppola
2,The Dark Knight,9.0,Christopher Nolan
3,The Godfather: Part II,9.0,Francis Ford Coppola
4,12 Angry Men,9.0,Sidney Lumet
5,The Lord of the Rings: The Return of the King,8.9,Peter Jackson
6,Pulp Fiction,8.9,Quentin Tarantino
7,Schindler's List,8.9,Steven Spielberg
8,Inception,8.8,Christopher Nolan
9,Fight Club,8.8,David Fincher


---
# ***6. Window Functions — rank without collapsing rows***
---

In [45]:
query = """
SELECT Series_Title, IMDB_Rating, Director,
       RANK() OVER (ORDER BY IMDB_Rating DESC) as rank
FROM movies
LIMIT 20
"""
pd.read_sql_query(query, conn)

,Series_Title,IMDB_Rating,Director,rank
0,The Shawshank Redemption,9.3,Frank Darabont,1
1,The Godfather,9.2,Francis Ford Coppola,2
2,The Dark Knight,9.0,Christopher Nolan,3
3,The Godfather: Part II,9.0,Francis Ford Coppola,3
4,12 Angry Men,9.0,Sidney Lumet,3
5,The Lord of the Rings: The Return of the King,8.9,Peter Jackson,6
6,Pulp Fiction,8.9,Quentin Tarantino,6
7,Schindler's List,8.9,Steven Spielberg,6
8,Inception,8.8,Christopher Nolan,9
9,Fight Club,8.8,David Fincher,9


In [47]:
query = """
SELECT Series_Title, Genre, IMDB_Rating, Director,
       RANK() OVER (PARTITION BY Genre ORDER BY IMDB_Rating DESC) as genre_rank
FROM movies
LIMIT 30
"""
pd.read_sql_query(query, conn)

,Series_Title,Genre,IMDB_Rating,Director,genre_rank
0,The Dark Knight Rises,"Action, Adventure",8.4,Christopher Nolan,1
1,Raiders of the Lost Ark,"Action, Adventure",8.4,Steven Spielberg,1
2,Batman Begins,"Action, Adventure",8.2,Christopher Nolan,3
3,Indiana Jones and the Last Crusade,"Action, Adventure",8.2,Steven Spielberg,3
4,First Blood,"Action, Adventure",7.7,Ted Kotcheff,5
5,"Aguirre, der Zorn Gottes","Action, Adventure, Biography",7.9,Werner Herzog,1
6,Sholay,"Action, Adventure, Comedy",8.2,Ramesh Sippy,1
7,The General,"Action, Adventure, Comedy",8.1,Clyde Bruckman,2
8,Bajrangi Bhaijaan,"Action, Adventure, Comedy",8.0,Kabir Khan,3
9,Guardians of the Galaxy,"Action, Adventure, Comedy",8.0,James Gunn,3


---
# ***SQL Mini Exercises***
---

---
# ***Exercise 1 — Movies where Gross > 100M***
---

In [54]:
query = """
SELECT Series_Title, IMDB_Rating, Gross
FROM movies
WHERE GROSS > 100000000
ORDER BY IMDB_Rating DESC
"""
result = pd.read_sql_query(query, conn)
print(f"Movies grossing over 100M : {len(result)}")
print(result.head(10).to_string(index=False))

Movies grossing over 100M : 187
                                     Series_Title  IMDB_Rating       Gross
                                    The Godfather          9.2 134966411.0
                                  The Dark Knight          9.0 534858444.0
    The Lord of the Rings: The Return of the King          8.9 377845905.0
                                     Pulp Fiction          8.9 107928762.0
                                        Inception          8.8 292576195.0
The Lord of the Rings: The Fellowship of the Ring          8.8 315544750.0
                                     Forrest Gump          8.8 330252182.0
            The Lord of the Rings: The Two Towers          8.7 342551365.0
                                       The Matrix          8.7 171479930.0
   Star Wars: Episode V - The Empire Strikes Back          8.7 290475067.0


---
# ***Exercise 2 — Directors with 3+ movies***
---

In [56]:
query = """
SELECT Director, COUNT(*) as movie_count
FROM movies
GROUP BY Director
HAVING movie_count >= 3
ORDER BY movie_count DESC
"""
result = pd.read_sql_query(query, conn)
print(f"Directors with 3+ movies in IMDb Top 1000 : {len(result)}")
print(result.to_string(index=False))

Directors with 3+ movies in IMDb Top 1000 : 97
             Director  movie_count
     Alfred Hitchcock           14
     Steven Spielberg           13
       Hayao Miyazaki           11
      Martin Scorsese           10
       Akira Kurosawa           10
          Woody Allen            9
      Stanley Kubrick            9
         Billy Wilder            9
    Quentin Tarantino            8
        David Fincher            8
       Clint Eastwood            8
    Christopher Nolan            8
           Rob Reiner            7
       Ingmar Bergman            7
         Howard Hawks            7
         Wes Anderson            6
         Sergio Leone            6
         Ridley Scott            6
    Richard Linklater            6
            Joel Coen            6
      Charles Chaplin            6
       Alfonso Cuarón            6
         Sidney Lumet            5
           Ron Howard            5
       Roman Polanski            5
      Robert Zemeckis            5
        

---
# ***Exercise 3 — Movies above average rating using subquery***
---

In [62]:
query = """
SELECT Series_Title, IMDB_Rating, Director
FROM movies
WHERE IMDB_Rating > (SELECT AVG(IMDB_Rating) FROM movies)
ORDER BY IMDB_Rating DESC
"""
result = pd.read_sql_query(query, conn)
avg_query = "SELECT ROUND(AVG(IMDB_Rating),3) as avg_rating FROM movies"
avg_rating = pd.read_sql_query(avg_query, conn).iloc[0, 0]
print(f"Overall average rating : {avg_rating}")
print(f"Movies above average   : {len(result)}")
print(f"That's {len(result)/10:.1f}%  of the top 1000\n")
print(result.head(10).to_string(index=False))

Overall average rating : 7.949
Movies above average   : 463
That's 46.3%  of the top 1000

                                 Series_Title  IMDB_Rating             Director
                     The Shawshank Redemption          9.3       Frank Darabont
                                The Godfather          9.2 Francis Ford Coppola
                              The Dark Knight          9.0    Christopher Nolan
                       The Godfather: Part II          9.0 Francis Ford Coppola
                                 12 Angry Men          9.0         Sidney Lumet
The Lord of the Rings: The Return of the King          8.9        Peter Jackson
                                 Pulp Fiction          8.9    Quentin Tarantino
                             Schindler's List          8.9     Steven Spielberg
                                    Inception          8.8    Christopher Nolan
                                   Fight Club          8.8        David Fincher
